# LULC - Train and predict

This notebook concatenates training data for many tiles for 2020, and trains a random forest model. It reads and writes to/from S3.

### Steps: 
1. Concat all tile CSV data from S3.
2. Filter outliers per class
3. Train model:
Try one model for all sites. (append all CSVs)
export model dump
in future we may need to make different models for different regions and year ranges.
train the model using the geomad of the year of the input products.

How many models do we need?
Pacific?
LS eras e.g. before 2013 and after.
Non-pacific model could be more diverse? might need more models.

In [ ]:
# Reload functions during development
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline

# Scientific core
from io import StringIO

import geopandas as gpd

# Machine learning
import joblib
import numpy as np
import pandas as pd

# Geospatial
import rioxarray  # noqa: F401
import zarr  # noqa: F401
from shapely import wkt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from ldn.training_data import PACIFIC_TRAINING_TILES
from ldn.typology import classes, colors
from ldn.utils import (
    CLASS_ATTR,
    MODEL_VERSION,
    TRAINING_DATA_VERSION,
    TRAINING_DATA_YEAR,
    WGS84,
    get_env_var,
)

region = "pacific"
year = TRAINING_DATA_YEAR

BUCKET = get_env_var("BUCKET")
print(f"BUCKET: {BUCKET}")

TRAINING_TILES = PACIFIC_TRAINING_TILES

print(f"Training tiles ({len(TRAINING_TILES)}): {TRAINING_TILES}")
print(f"Training data version: {TRAINING_DATA_VERSION}")
print(f"Model version: {MODEL_VERSION}")
print(f"Year: {year}")

In [ ]:
# Gather all of the CSVs of training points for many AOIs.
# paths = f"training_data/{TRAINING_DATA_VERSION}/{tile_id}/{year}/samples_*.csv"
# files = glob.glob(paths)
import os

import boto3

# from ldn.aws import s3_client
from ldn.utils import parse_tile_id

aws_profile = os.environ.get("AWS_PROFILE")
# %env AWS_PROFILE
print(f"Using AWS profile: {aws_profile}")

aws_session = boto3.Session(profile_name=aws_profile)
print(f"Using AWS profile: {aws_session.profile_name}, region: {aws_session.region_name}")
s3_client = aws_session.client("s3", region_name=aws_session.region_name)

# List all training CSVs for this version
resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=f"training_data/{TRAINING_DATA_VERSION}/")
# This gets all regions (Pacific and non-Pacific). Can easily split by region here.
keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith("samples.csv")]
num_files_found = len(keys)
print(f"Found {num_files_found} CSV files")

# Filter to just the training tiles.
filtered_keys = []
for id, region, country_dict in TRAINING_TILES:
    tile_id_x, tile_id_y = parse_tile_id(id)
    for key in keys:
        if f"/{tile_id_x:03d}/{tile_id_y:03d}/" in key:  # Zero-padding tile index values.
            filtered_keys.append(key)

num_files_found = len(filtered_keys)
print(f"Filtered to {num_files_found} CSV files for training tiles")
print(filtered_keys)

expected_num_files = len(TRAINING_TILES)
assert num_files_found == expected_num_files, (
    f"There should be {expected_num_files} training CSV files found in S3, not {num_files_found}."
)

# Stream each CSV from S3 into a DataFrame
dfs = {}
for key in filtered_keys:
    print(f"  Loading s3://{BUCKET}/{key}")
    body = s3_client.get_object(Bucket=BUCKET, Key=key)["Body"].read().decode("utf-8")
    tile_id_x = key.split("/")[-4]
    tile_id_y = key.split("/")[-3]
    print(tile_id_x, tile_id_y)
    dfs[(tile_id_x, tile_id_y)] = pd.read_csv(StringIO(body))
samples = pd.concat(dfs.values(), ignore_index=True)
print(f"Combined training data: {len(samples)} samples from {len(dfs)} files")

# Parse WKT geometry so we can do spatial region assignment
samples["geometry"] = samples["geometry"].apply(wkt.loads)
samples = gpd.GeoDataFrame(samples, geometry="geometry", crs=WGS84)

samples.drop(columns=["outlier"], inplace=True, errors="ignore")
# samples.drop(columns=["geometry"], inplace=True, errors="ignore")
samples.drop(columns=["time"], inplace=True, errors="ignore")
samples.drop(columns=["spatial_ref"], inplace=True, errors="ignore")
print(samples.columns)

print(samples.head())
print("Per class counts:")
print(samples[CLASS_ATTR].value_counts().sort_index())

# Tiles are often missing one or more classes e.g. Singapore is missing 4:Wetland.

In [ ]:
import re

from matplotlib.colors import ListedColormap


def rgb_str_to_hex(rgb_str):
    r, g, b = map(int, re.findall(r"\d+", rgb_str))
    return "#{:02x}{:02x}{:02x}".format(r, g, b)


# Build ordered hex colors for classes 1-7
hex_colors = [rgb_str_to_hex(colors[i]) for i in range(1, 8)]

samples["lulc"] = samples["lulc"].astype("category")

samples.explore(
    column="lulc",
    categorical=True,
    cmap=ListedColormap(hex_colors),
)

### Visualise class pixel counts per tile

In [ ]:
import math

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

ncols = 8
nrows = math.ceil(num_files_found / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), sharey=False)
axes = np.array(axes).flatten()

hex_colors2 = {cls: rgb_str_to_hex(rgb) for cls, rgb in colors.items()}

global_max = max(df[CLASS_ATTR].value_counts().max() for df in dfs.values())

for ax, (tile_id, df) in zip(axes, dfs.items()):
    counts = df[CLASS_ATTR].value_counts().sort_index()
    bar_colors = [hex_colors2.get(cls, "#808080") for cls in counts.index]

    ax.bar(
        counts.index.astype(str),
        counts.values,
        color=bar_colors,
        edgecolor="black",
        linewidth=0.5,
    )

    ax.set_ylim(0, global_max * 1.05)  # 5% headroom

    country_name = next(
        (list(t[2].keys())[0] for t in TRAINING_TILES if t[0] == "_".join(tile_id)),
        "Unknown",
    )

    ax.set_title(f"{tile_id} ({country_name})", fontsize=15)
    ax.set_xlabel("class")
    ax.set_ylabel("count")

# Hide unused axes
for ax in axes[num_files_found:]:
    ax.set_visible(False)

# Shared legend
handles = [mpatches.Patch(color=hex_colors2[cls], label=str(cls), edgecolor="black") for cls in hex_colors2]
fig.legend(
    handles=handles,
    title="class",
    loc="lower center",
    ncol=len(colors),
    bbox_to_anchor=(0.5, -0.05),
)

plt.tight_layout()
plt.show()

In [ ]:
# from shapely.geometry import box

# # Define regions to split models by.
# max_y = 90
# min_y = -90

# split_x_1 = -180
# split_x_2 = -107
# split_x_3 = -40
# # split_x_4 = 90 # Puts Singapore and Timor-Leste in Pacific2
# split_x_4 = 128  # Puts Singapore and Timor-Leste in Africa
# split_x_5 = 180

# regions = {
#     # West to East.
#     "pacific1": box(split_x_1, min_y, split_x_2, max_y),
#     "caribbean": box(split_x_2, min_y, split_x_3, max_y),
#     "africa": box(split_x_3, min_y, split_x_4, max_y),
#     "pacific2": box(split_x_4, min_y, split_x_5, max_y),
# }

# # Build a GeoDataFrame of region boxes and spatial-join to samples
# regions_gdf = gpd.GeoDataFrame(
#     {"region": regions.keys()},
#     geometry=list(regions.values()),
#     crs=WGS84,
# )
# samples = gpd.sjoin(
#     samples, regions_gdf[["region", "geometry"]], how="left", predicate="intersects"
# )
# samples.drop(columns=["index_right"], inplace=True, errors="ignore")

# print("Samples per region:")
# print(samples["region"].value_counts())
# print(f"Samples without a region: {samples['region'].isna().sum()}")

# # TODO: Use this region logic once we split the model per region.
# # TODO: Count classes per region. Many will be missing.

In [ ]:
# # Make different models for time periods.
# years = [year for year in range(2000, 2025)]  # 2000 to 2024
# print(years)

# # TODO: Verify these splits.
# # In our GeoMAD logic we use <= LS7_YEAR_THRESHOLD as a condition to use a buffered year and T1 and T2 data.
# time_periods = {
#     "tm_etm": [year for year in range(2000, 2003)],  # L5 + L7 pre-SLC failure
#     "slc_off": [year for year in range(2003, 2013)],  # L7 SLC-off, L5 aging/gone
#     "oli": [year for year in range(2013, 2025)],  # L8/L9, modern radiometry
# }
# print(time_periods)

In [ ]:
# # TODO: Cross regions and periods
# region_periods = {}

## Again filter for outliers (for the combined data for many tiles).

In [ ]:
# # TODO here balance concatenated training data by class e.g.
# if one class has a huge amount of samples (after contatenation).
# # TODO: in future think about region-splitting per model.
# # TODO: Do a sample distribution check per class AND country/region.
# # This can decide which countries should be included to train the model/model per region.
# # Focus on getting all classes e.g. other.

# TODO: Use ldn/training_data.py's filter_outliers function.
# from ldn.training_data import filter_outliers
# TODO: Run a baseline version and then test if outlier filtering improves the model and classification.

### Test correlation between features. Exclude >95% correlated features.

In [ ]:
# exclude_cols = ['lulc', 'geometry']
# feature_cols = [c for c in samples.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(samples[c])]

# corr = samples[feature_cols].corr().abs()

# # Upper triangle mask (ignore self-correlation on the diagonal)
# upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

# # Find pairs above 0.95
# high_corr_pairs = [
#     (col, row, upper.loc[row, col])
#     for col in upper.columns
#     for row in upper.index
#     if upper.loc[row, col] > 0.95
# ]

# if high_corr_pairs:
#     print("Highly correlated feature pairs (>0.95):")
#     for a, b, r in sorted(high_corr_pairs, key=lambda x: -x[2]):
#         print(f"  {a:12s} ↔ {b:12s}  r={r:.3f}")

#     # Drop the second feature in each pair (keep first encountered)
#     to_drop = {b for _, b, _ in high_corr_pairs}
#     print(f"\nDropping {len(to_drop)} redundant feature(s): {to_drop}")
#     samples.drop(columns=to_drop, inplace=True, errors='ignore')
# else:
#     print("No feature pairs with correlation > 0.95 — keeping all features.")

# # Heatmap using matplotlib only
# fig, ax = plt.subplots(figsize=(12, 10))
# im = ax.imshow(corr.values, cmap="coolwarm", vmin=0, vmax=1)
# ax.set_xticks(range(len(feature_cols)))
# ax.set_yticks(range(len(feature_cols)))
# ax.set_xticklabels(feature_cols, rotation=45, ha="right", fontsize=7)
# ax.set_yticklabels(feature_cols, fontsize=7)
# for i in range(len(feature_cols)):
#     for j in range(len(feature_cols)):
#         ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=5,
#                 color="white" if corr.values[i, j] > 0.7 else "black")
# plt.colorbar(im, ax=ax, label="Absolute correlation")
# ax.set_title("Feature Correlation (absolute)")
# plt.tight_layout()
# plt.show()

## Train the model

In [ ]:
# Split 70/30 into train/validation. Splits the classes into train/validation in a representative way.
train_gdf, validation_gdf = train_test_split(samples, test_size=0.3, stratify=samples[CLASS_ATTR], random_state=42)

print(f"Training set class distribution:\n{train_gdf[CLASS_ATTR].value_counts()}")
print(f"Validation set class distribution:\n{validation_gdf[CLASS_ATTR].value_counts()}")
print(train_gdf)

# Define features and target
feature_cols = [
    c for c in train_gdf.columns if c != CLASS_ATTR and c != "region" and pd.api.types.is_numeric_dtype(train_gdf[c])
]

# Pass DataFrames (not .values) so scikit-learn records feature names.
# This avoids the "fitted without feature names" warning at prediction time.
X_train = train_gdf[feature_cols]
y_train = train_gdf[CLASS_ATTR]
X_test = validation_gdf[feature_cols]
y_test = validation_gdf[CLASS_ATTR]

print(f"Classes: {np.unique(y_train)}")

# 500 estimators could be needed for a many country model.
classifier = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
model = classifier.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Feature importance
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances)
# TODO: Drop noisy features and retrain.
# Aaspect could be less important because of low seasonailty in equatorial regions.

present = np.unique(np.concatenate([y_test, y_pred]))
target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v in present]
print(f"Target names: {target_names}")

print(classification_report(y_test, y_pred, target_names=target_names, labels=present))

In [ ]:
# One method of validation is for example to train on all countries in the Caribbean, and then predict for the last one.

# Make training data for Cape Verde, but don't use that to train the model. Just use it to test. The below current
# validation overpromises accuracy.
# Hold out testing.
# Cross-validation testing.
# Cape Verde is mostly bare land = other. SR is really bright compared to Singapore and Fiji.

# Validate on single year data and change calculations over years.
# E.g. a pixel could flip between grass and crop year to year just due to randomness in the model's training data.
# RF Classification Report - check the confidence map - how many trees voted the same way. 50% confidence is standard.
# Post-processing can be done e.g. we smooth these temporal flips to one class over time.
# Post-processing should be minimal.
# Another post-processing method is to smooth any single pixel surrounded in space or time by another
# class to that same class.

In [ ]:
from pathlib import Path

from ldn.utils import MODEL_VERSION, get_pathstyle_url_base

s3_key = f"models/{MODEL_VERSION}/{region}/{year}/lulc_random_forest_model_{region}_{year}.joblib"

joblib_path = Path(f"../../ldn/{s3_key}")
joblib_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, joblib_path)

# Write the model to S3 so it can be loaded in prediction step.
s3_client.upload_file(str(joblib_path), BUCKET, s3_key)

s3_url = f"{get_pathstyle_url_base(BUCKET)}/{s3_key}"
print(f"Uploaded model to {s3_url}")

# # Load the model
# model = joblib.load(joblib_path)

In [ ]:
# TODO: Use MODEL_TEST_TILES as a test set that is not used to train the model. E.g. one or more whole tiles.

# MODEL_TEST_TILES is used for testing different versions of the model. This provides a consistent test across versions.
# validation_gdf is used for testing a single version (as it is created).

If classes are confused e.g. crop and grassland, then increase the strictness of agreement needed for that class.

Will the model be released? If yes, Wei Ji reccomends another format than joblib. It is more library-agnostic.